# Load responses and use Perspective API to score them

In [1]:
from pathlib import Path
import pandas as pd
from typing import Optional, List, Iterable, Dict, Any
from tqdm import tqdm
from utils.perspective_api_modified import PerspectiveWorker, unpack_scores
from utils.utils import load_jsonl, batchify
from utils.constants import PERSPECTIVE_API_ATTRIBUTES_LOWER

In [2]:
output_dir = "../Datasets/RealToxicityPrompts-Output/gpt2/"
output_dir = Path(output_dir)
generations_file = output_dir / 'generations.jsonl'
perspective_file = output_dir / 'perspective.jsonl'
output_file = output_dir / 'gens_gpt2.jsonl'

def make_generations_col(generations, responses):
    for generation, response in zip(generations, responses):
        if response['response']:
            response = unpack_scores(response['response'])[0]
        else:
            response = {x: None for x in PERSPECTIVE_API_ATTRIBUTES_LOWER}
        yield {'text': generation, **response}

def collate(dataset: Optional[pd.DataFrame], generations: List[str], responses: Iterable[Dict[str, Any]],
            output_file: str):
    generations_col_iter = make_generations_col(generations, responses)
    if dataset is None:
        generations_col = list(tqdm(generations_col_iter, total=len(list(load_jsonl(generations_file))), desc='Collating files'))
        dataset = pd.DataFrame(generations_col)
    else:
        assert len(generations) % len(dataset) == 0
        n = len(generations) // len(dataset)
        print(f"Detected samples per prompt:", n)
        generations_col = list(tqdm(batchify(generations_col_iter, n), total=len(dataset), desc='Collating files'))
        dataset['generations'] = generations_col

    dataset.to_json(output_file, orient='records', lines=True)

# Create perspective worker thread
perspective = PerspectiveWorker(out_file=perspective_file,
                                total=len(list(load_jsonl(generations_file))),
                                rate_limit=1)

# Generate and collate perspective scores
generations = load_jsonl(generations_file)
for i, gen in enumerate(generations):
    perspective(f'generation-{i}', gen)


perspective.stop()
print('Finished perspective scoring!')

print('Collating output files')
collate(None, generations, load_jsonl(perspective_file), output_file)

print("Perspective scores saved to gens_gpt2.jsonl")

Waiting for Perspective to finish...
Finished perspective scoring!
Collating output files


Collating files:   0%|          | 0/8 [00:00<?, ?it/s]

Perspective scores saved to gens_gpt2.jsonl
